# 06 评分卡与 Cutoff 分析（Phase 6）

这个 notebook 负责把 `Phase 5` 的候选模型输出翻译成 **score / risk band / cutoff** 语言，并显式把 proxy-sensitive uplift 与 grouped governance diagnostic 一起带入策略分析。

注意：仓库内的 `credit_visable.scoring.pdo_scorecard` 仍然是 placeholder。本 notebook 采用 **notebook-local** 的分析级分数映射，不应描述为 production-ready PDO scorecard。


## 1. 运行参数与辅助函数

本节定义 Phase 6 的默认评分参数、风险分层、cutoff 策略、可视化函数，以及写出 artifact 所需的本地辅助函数。


In [ ]:
from pathlib import Path
import json
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.metrics import brier_score_loss

root = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / "pyproject.toml").exists()),
    Path.cwd(),
)
src = root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

warnings.filterwarnings(
    "ignore",
    message="Pandas requires version '1.3.6' or newer of 'bottleneck'",
)
sns.set_theme(style="whitegrid")

from credit_visable.config import load_settings
from credit_visable.scoring import build_scorecard_placeholder
from credit_visable.utils import apply_report_style, get_paths

apply_report_style()

settings = load_settings()
paths = get_paths()

PHASE_5_OUTPUT_ROOT = paths.data_processed / "xai_fairness"
PHASE_6_OUTPUT_ROOT = paths.data_processed / "scorecard_cutoff"
FIGURE_OUTPUT_DIR = paths.reports_figures

BASE_SCORE = 600
POINTS_TO_DOUBLE_ODDS = 20
BASE_ODDS = 50.0
PD_EPSILON = 1e-6
CALIBRATION_BINS = 10
AGE_BAND_ORDER = ["[20,25)", "[25,35)", "[35,45)", "[45,55)", "[55,65)", "[65,70)", "Missing"]
RISK_BAND_ORDER = ["A", "B", "C", "D", "E"]
DECISION_ORDER = ["approve", "review", "reject"]
GROUP_REVIEW_COLUMNS = ["age_band", "family_status_group", "region_rating_group"]
POLICY_SCENARIOS = {
    "conservative": {"approve_cutoff": 640, "review_cutoff": 600},
    "balanced": {"approve_cutoff": 620, "review_cutoff": 580},
    "growth": {"approve_cutoff": 600, "review_cutoff": 560},
}

PHASE_6_READY = False
phase6_load_ok = False
phase6_score_ok = False
phase6_policy_ok = False

placeholder_metadata = build_scorecard_placeholder(
    base_score=BASE_SCORE,
    base_odds=BASE_ODDS,
    points_to_double_odds=POINTS_TO_DOUBLE_ODDS,
)
phase5_selection_payload = {}
phase5_summary_payload = {}
candidate_model = {}
comparator_model = {}
score_transform_meta = {}
score_cutoff_grid = []
figure_paths = {}

score_input_frame = pd.DataFrame()
score_frame = pd.DataFrame()
score_decile_summary = pd.DataFrame()
risk_band_summary = pd.DataFrame()
calibration_summary = pd.DataFrame()
cutoff_sweep = pd.DataFrame()
policy_scenarios = pd.DataFrame()
policy_group_summary = pd.DataFrame()
score_migration_matrix = pd.DataFrame()
decision_migration_matrix = pd.DataFrame()
validation_checks = pd.DataFrame()


def _to_builtin(value):
    if isinstance(value, dict):
        return {str(key): _to_builtin(item) for key, item in value.items()}
    if isinstance(value, list):
        return [_to_builtin(item) for item in value]
    if isinstance(value, tuple):
        return [_to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return [_to_builtin(item) for item in value.tolist()]
    if isinstance(value, (np.floating, float)):
        if not np.isfinite(value):
            return None
        return float(value)
    if isinstance(value, (np.integer, int)):
        return int(value)
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if pd.isna(value):
        return None
    return value


def clip_probabilities(values, epsilon: float = PD_EPSILON) -> np.ndarray:
    return np.clip(np.asarray(values, dtype=float), epsilon, 1.0 - epsilon)


def build_score_transform_meta(
    base_score: float = BASE_SCORE,
    points_to_double_odds: float = POINTS_TO_DOUBLE_ODDS,
    base_odds: float = BASE_ODDS,
) -> dict[str, float | str | bool]:
    factor = float(points_to_double_odds / np.log(2.0))
    offset = float(base_score - factor * np.log(base_odds))
    return {
        "base_score": float(base_score),
        "points_to_double_odds": float(points_to_double_odds),
        "base_odds": float(base_odds),
        "factor": factor,
        "offset": offset,
        "calibration_fitted": False,
        "score_formula": "score = base_score + factor * ln(((1-p)/p) / base_odds)",
        "notes": "Notebook-local analysis score transform only. Repository PDO module remains placeholder-only.",
    }


def odds_to_score(odds, meta: dict[str, float | str | bool]) -> np.ndarray:
    odds_array = np.asarray(odds, dtype=float)
    return meta["base_score"] + meta["factor"] * np.log(odds_array / meta["base_odds"])


def pd_to_score(pd_values, meta: dict[str, float | str | bool], name: str | None = None) -> pd.Series:
    clipped = clip_probabilities(pd_values)
    odds = (1.0 - clipped) / clipped
    scores = odds_to_score(odds, meta)
    if isinstance(pd_values, pd.Series):
        return pd.Series(scores, index=pd_values.index, name=name or pd_values.name)
    return pd.Series(scores, name=name)


def assign_risk_band(score_values: pd.Series) -> pd.Series:
    numeric_scores = pd.to_numeric(score_values, errors="coerce")
    bands = np.select(
        [
            numeric_scores >= 680,
            numeric_scores >= 640,
            numeric_scores >= 600,
            numeric_scores >= 560,
        ],
        ["A", "B", "C", "D"],
        default="E",
    )
    return pd.Series(bands, index=score_values.index, name="risk_band")


def assign_score_decile(score_values: pd.Series, num_bins: int = 10) -> pd.Series:
    numeric_scores = pd.to_numeric(score_values, errors="coerce")
    valid_mask = numeric_scores.notna()
    decile = pd.Series(pd.NA, index=score_values.index, dtype="Int64", name="risk_decile")
    if valid_mask.sum() == 0:
        return decile

    ranked = numeric_scores.loc[valid_mask].rank(method="first", ascending=True)
    bin_count = min(num_bins, int(valid_mask.sum()))
    if bin_count <= 1:
        decile.loc[valid_mask] = 1
        return decile

    labels = list(range(1, bin_count + 1))
    decile.loc[valid_mask] = pd.qcut(ranked, q=bin_count, labels=labels).astype("Int64")
    return decile


def build_score_cutoff_grid(score_values: pd.Series, step: int = 10) -> list[int]:
    numeric_scores = pd.to_numeric(score_values, errors="coerce").dropna()
    if numeric_scores.empty:
        return []

    lower = int(np.floor(np.nanpercentile(numeric_scores, 1) / step) * step)
    upper = int(np.ceil(np.nanpercentile(numeric_scores, 99) / step) * step)
    if upper < lower:
        lower, upper = upper, lower
    if lower == upper:
        return [lower]
    return list(range(lower, upper + step, step))


def apply_policy(score_values: pd.Series, approve_cutoff: float, review_cutoff: float) -> pd.Series:
    numeric_scores = pd.to_numeric(score_values, errors="coerce")
    decisions = np.select(
        [numeric_scores >= approve_cutoff, numeric_scores >= review_cutoff],
        ["approve", "review"],
        default="reject",
    )
    return pd.Series(decisions, index=score_values.index, name="policy_decision")


def summarize_score_deciles(
    frame: pd.DataFrame,
    decile_column: str,
    score_column: str,
    pd_column: str,
    target_column: str,
) -> pd.DataFrame:
    working = frame[[decile_column, score_column, pd_column, target_column]].copy()
    working = working.dropna(subset=[decile_column])
    if working.empty:
        return pd.DataFrame()

    summary = (
        working.groupby(decile_column, dropna=False, observed=False)
        .agg(
            count=(target_column, "size"),
            bad_count=(target_column, "sum"),
            actual_default_rate=(target_column, "mean"),
            mean_predicted_pd=(pd_column, "mean"),
            mean_score=(score_column, "mean"),
            min_score=(score_column, "min"),
            max_score=(score_column, "max"),
        )
        .reset_index()
        .rename(columns={decile_column: "risk_decile"})
        .sort_values("risk_decile")
        .reset_index(drop=True)
    )
    summary["population_share"] = summary["count"] / len(working)
    summary["risk_decile"] = summary["risk_decile"].astype("Int64")
    return summary[
        [
            "risk_decile",
            "count",
            "population_share",
            "bad_count",
            "actual_default_rate",
            "mean_predicted_pd",
            "mean_score",
            "min_score",
            "max_score",
        ]
    ]


def summarize_risk_bands(
    frame: pd.DataFrame,
    band_column: str,
    score_column: str,
    pd_column: str,
    target_column: str,
) -> pd.DataFrame:
    working = frame[[band_column, score_column, pd_column, target_column]].copy()
    working[band_column] = working[band_column].astype("object").fillna("Missing")
    if working.empty:
        return pd.DataFrame()

    summary = (
        working.groupby(band_column, dropna=False, observed=False)
        .agg(
            count=(target_column, "size"),
            bad_count=(target_column, "sum"),
            actual_default_rate=(target_column, "mean"),
            mean_predicted_pd=(pd_column, "mean"),
            mean_score=(score_column, "mean"),
            min_score=(score_column, "min"),
            max_score=(score_column, "max"),
        )
        .reset_index()
        .rename(columns={band_column: "risk_band"})
    )
    summary["population_share"] = summary["count"] / len(working)
    summary["band_rank"] = summary["risk_band"].map({band: rank for rank, band in enumerate(RISK_BAND_ORDER)})
    summary = summary.sort_values(["band_rank", "risk_band"]).reset_index(drop=True)
    return summary[
        [
            "risk_band",
            "count",
            "population_share",
            "bad_count",
            "actual_default_rate",
            "mean_predicted_pd",
            "mean_score",
            "min_score",
            "max_score",
        ]
    ]


def build_calibration_summary(
    frame: pd.DataFrame,
    pd_column: str,
    target_column: str,
    score_column: str,
    bins: int = CALIBRATION_BINS,
) -> pd.DataFrame:
    working = frame[[pd_column, target_column, score_column]].copy()
    working[pd_column] = pd.to_numeric(working[pd_column], errors="coerce")
    working[target_column] = pd.to_numeric(working[target_column], errors="coerce")
    working = working.dropna(subset=[pd_column, target_column])
    if working.empty:
        return pd.DataFrame()

    bin_count = min(bins, len(working))
    working["calibration_bin"] = 1
    if bin_count > 1:
        ranked = working[pd_column].rank(method="first", ascending=True)
        working["calibration_bin"] = pd.qcut(ranked, q=bin_count, labels=list(range(1, bin_count + 1)))

    summary = (
        working.groupby("calibration_bin", dropna=False, observed=False)
        .agg(
            count=(target_column, "size"),
            bad_count=(target_column, "sum"),
            actual_default_rate=(target_column, "mean"),
            mean_predicted_pd=(pd_column, "mean"),
            mean_score=(score_column, "mean"),
        )
        .reset_index()
        .sort_values("calibration_bin")
        .reset_index(drop=True)
    )
    summary["calibration_bin"] = summary["calibration_bin"].astype("Int64")
    summary["population_share"] = summary["count"] / len(working)
    summary["calibration_gap"] = summary["actual_default_rate"] - summary["mean_predicted_pd"]
    brier = float(brier_score_loss(working[target_column], working[pd_column]))
    summary["brier_score"] = brier
    return summary[
        [
            "calibration_bin",
            "count",
            "population_share",
            "bad_count",
            "actual_default_rate",
            "mean_predicted_pd",
            "calibration_gap",
            "mean_score",
            "brier_score",
        ]
    ]


def build_cutoff_sweep(
    frame: pd.DataFrame,
    score_column: str,
    pd_column: str,
    target_column: str,
    cutoff_grid: list[int],
) -> pd.DataFrame:
    rows = []
    total_bad = float(frame[target_column].sum())
    for cutoff in cutoff_grid:
        approved_mask = frame[score_column] >= cutoff
        rejected_mask = ~approved_mask
        approved = frame.loc[approved_mask]
        rejected = frame.loc[rejected_mask]
        rows.append(
            {
                "score_cutoff": int(cutoff),
                "approved_count": int(approved_mask.sum()),
                "rejected_count": int(rejected_mask.sum()),
                "approval_rate": float(approved_mask.mean()),
                "reject_rate": float(rejected_mask.mean()),
                "approved_bad_rate": float(approved[target_column].mean()) if not approved.empty else np.nan,
                "rejected_bad_capture_rate": float(rejected[target_column].sum() / total_bad) if total_bad > 0 else 0.0,
                "approved_mean_predicted_pd": float(approved[pd_column].mean()) if not approved.empty else np.nan,
                "approved_mean_score": float(approved[score_column].mean()) if not approved.empty else np.nan,
                "rejected_mean_predicted_pd": float(rejected[pd_column].mean()) if not rejected.empty else np.nan,
                "rejected_mean_score": float(rejected[score_column].mean()) if not rejected.empty else np.nan,
            }
        )
    return pd.DataFrame(rows)


def build_policy_summary(
    frame: pd.DataFrame,
    scenario_name: str,
    decision_column: str,
    score_column: str,
    pd_column: str,
    target_column: str,
    approve_cutoff: float,
    review_cutoff: float,
) -> dict[str, float | int | str | None]:
    total_bad = float(frame[target_column].sum())
    decisions = frame[decision_column].astype(str)
    approved = frame.loc[decisions == "approve"]
    review = frame.loc[decisions == "review"]
    rejected = frame.loc[decisions == "reject"]
    return {
        "scenario_name": scenario_name,
        "approve_cutoff": float(approve_cutoff),
        "review_cutoff": float(review_cutoff),
        "approved_count": int(len(approved)),
        "review_count": int(len(review)),
        "rejected_count": int(len(rejected)),
        "approval_rate": float((decisions == "approve").mean()),
        "review_rate": float((decisions == "review").mean()),
        "reject_rate": float((decisions == "reject").mean()),
        "approved_bad_rate": float(approved[target_column].mean()) if not approved.empty else np.nan,
        "review_bad_rate": float(review[target_column].mean()) if not review.empty else np.nan,
        "reject_bad_rate": float(rejected[target_column].mean()) if not rejected.empty else np.nan,
        "rejected_bad_capture_rate": float(rejected[target_column].sum() / total_bad) if total_bad > 0 else 0.0,
        "review_bad_capture_rate": float(review[target_column].sum() / total_bad) if total_bad > 0 else 0.0,
        "mean_predicted_pd": float(frame[pd_column].mean()),
        "mean_score": float(frame[score_column].mean()),
        "approved_mean_predicted_pd": float(approved[pd_column].mean()) if not approved.empty else np.nan,
        "review_mean_predicted_pd": float(review[pd_column].mean()) if not review.empty else np.nan,
        "rejected_mean_predicted_pd": float(rejected[pd_column].mean()) if not rejected.empty else np.nan,
    }


def build_policy_group_summary(
    frame: pd.DataFrame,
    group_columns: list[str],
    scenario_name: str,
    decision_column: str,
    score_column: str,
    pd_column: str,
    target_column: str,
) -> pd.DataFrame:
    outputs = []
    for group_column in group_columns:
        working = frame[[group_column, decision_column, score_column, pd_column, target_column]].copy()
        working[group_column] = working[group_column].astype("object").fillna("Missing")
        working["approved_flag"] = (working[decision_column] == "approve").astype(int)
        working["review_flag"] = (working[decision_column] == "review").astype(int)
        working["rejected_flag"] = (working[decision_column] == "reject").astype(int)
        working["approved_bad_flag"] = np.where(
            working[decision_column] == "approve",
            working[target_column],
            np.nan,
        )
        summary = (
            working.groupby(group_column, dropna=False, observed=False)
            .agg(
                count=(target_column, "size"),
                bad_count=(target_column, "sum"),
                actual_default_rate=(target_column, "mean"),
                mean_predicted_pd=(pd_column, "mean"),
                mean_score=(score_column, "mean"),
                approval_rate=("approved_flag", "mean"),
                review_rate=("review_flag", "mean"),
                reject_rate=("rejected_flag", "mean"),
                approved_count=("approved_flag", "sum"),
                approved_bad_rate=("approved_bad_flag", "mean"),
            )
            .reset_index()
            .rename(columns={group_column: "group"})
        )
        summary["population_share"] = summary["count"] / len(working)
        summary.insert(0, "scenario_name", scenario_name)
        summary.insert(1, "protected_attribute", group_column)
        outputs.append(summary)

    if not outputs:
        return pd.DataFrame()

    return pd.concat(outputs, ignore_index=True)


def build_migration_matrix(
    frame: pd.DataFrame,
    from_column: str,
    to_column: str,
    count_column_name: str = "count",
) -> pd.DataFrame:
    working = frame[[from_column, to_column]].copy()
    working[from_column] = working[from_column].astype("object").fillna("Missing")
    working[to_column] = working[to_column].astype("object").fillna("Missing")
    return (
        working.groupby([from_column, to_column], dropna=False, observed=False)
        .size()
        .reset_index(name=count_column_name)
    )


def _sort_group_metric_frame(summary_frame: pd.DataFrame, protected_attribute: str) -> pd.DataFrame:
    plot_frame = summary_frame.copy()
    if plot_frame.empty:
        return plot_frame
    if protected_attribute == "age_band":
        plot_frame["group_order"] = plot_frame["group"].map(
            {label: order for order, label in enumerate(AGE_BAND_ORDER)}
        )
        plot_frame = plot_frame.sort_values(["group_order", "group"])
    else:
        numeric_group = pd.to_numeric(plot_frame["group"], errors="coerce")
        if numeric_group.notna().all():
            plot_frame["group_order"] = numeric_group
            plot_frame = plot_frame.sort_values(["group_order", "group"])
        else:
            plot_frame = plot_frame.sort_values(["count", "group"], ascending=[False, True])
    return plot_frame.reset_index(drop=True)


def save_score_histogram(frame: pd.DataFrame, column: str, title: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(8, 4.8))
    if frame.empty or column not in frame.columns:
        ax.text(0.5, 0.5, "No data available", ha="center", va="center")
        ax.axis("off")
    else:
        sns.histplot(data=frame, x=column, bins=35, kde=True, color="#4C78A8", ax=ax)
        ax.set_xlabel("Score")
        ax.set_ylabel("Count")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def save_ecdf_curve(frame: pd.DataFrame, column: str, title: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(8, 4.8))
    values = pd.to_numeric(frame.get(column, pd.Series(dtype=float)), errors="coerce").dropna().sort_values()
    if values.empty:
        ax.text(0.5, 0.5, "No data available", ha="center", va="center")
        ax.axis("off")
    else:
        cumulative = np.arange(1, len(values) + 1) / len(values)
        ax.plot(values.to_numpy(), cumulative, color="#4C78A8", linewidth=2)
        ax.set_xlabel("Score")
        ax.set_ylabel("Cumulative Share")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def save_pd_to_score_curve(meta: dict[str, float | str | bool], title: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(8, 4.8))
    pd_grid = np.linspace(0.001, 0.999, 500)
    score_grid = pd_to_score(pd_grid, meta)
    ax.plot(pd_grid, score_grid, color="#E45756", linewidth=2)
    ax.set_xlabel("Predicted PD")
    ax.set_ylabel("Score")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def save_calibration_curve(summary_frame: pd.DataFrame, title: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(6, 6))
    if summary_frame.empty:
        ax.text(0.5, 0.5, "No data available", ha="center", va="center")
        ax.axis("off")
    else:
        ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect calibration")
        ax.plot(
            summary_frame["mean_predicted_pd"],
            summary_frame["actual_default_rate"],
            marker="o",
            color="#4C78A8",
            linewidth=2,
            label="Observed by quantile bin",
        )
        ax.set_xlabel("Mean Predicted PD")
        ax.set_ylabel("Actual Default Rate")
        ax.legend(loc="upper left")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def save_decile_reliability_chart(summary_frame: pd.DataFrame, title: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(9, 4.8))
    if summary_frame.empty:
        ax.text(0.5, 0.5, "No data available", ha="center", va="center")
        ax.axis("off")
    else:
        x = np.arange(len(summary_frame))
        width = 0.35
        ax.bar(x - width / 2, summary_frame["mean_predicted_pd"], width=width, color="#4C78A8", label="Mean predicted PD")
        ax.bar(x + width / 2, summary_frame["actual_default_rate"], width=width, color="#E45756", label="Actual default rate")
        ax.set_xticks(x)
        ax.set_xticklabels(summary_frame["risk_decile"].astype(str))
        ax.set_xlabel("Risk Decile (1 = highest risk)")
        ax.set_ylabel("Rate")
        ax.legend(loc="best")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def save_risk_band_metric_chart(
    summary_frame: pd.DataFrame,
    metric_column: str,
    title: str,
    path: Path,
    color: str = "#4C78A8",
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(8, 4.8))
    if summary_frame.empty:
        ax.text(0.5, 0.5, "No data available", ha="center", va="center")
        ax.axis("off")
    else:
        plot_frame = summary_frame.copy()
        plot_frame["band_rank"] = plot_frame["risk_band"].map({band: rank for rank, band in enumerate(RISK_BAND_ORDER)})
        plot_frame = plot_frame.sort_values(["band_rank", "risk_band"])
        ax.bar(plot_frame["risk_band"], plot_frame[metric_column], color=color)
        ax.set_xlabel("Risk Band")
        ax.set_ylabel(metric_column.replace("_", " ").title())
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def save_cutoff_metric_chart(
    summary_frame: pd.DataFrame,
    metric_column: str,
    title: str,
    path: Path,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(8, 4.8))
    if summary_frame.empty:
        ax.text(0.5, 0.5, "No data available", ha="center", va="center")
        ax.axis("off")
    else:
        ax.plot(summary_frame["score_cutoff"], summary_frame[metric_column], marker="o", color="#4C78A8")
        ax.set_xlabel("Score Cutoff")
        ax.set_ylabel(metric_column.replace("_", " ").title())
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def save_policy_composition_chart(summary_frame: pd.DataFrame, title: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(8, 4.8))
    if summary_frame.empty:
        ax.text(0.5, 0.5, "No data available", ha="center", va="center")
        ax.axis("off")
    else:
        plot_frame = summary_frame.set_index("scenario_name")[["approval_rate", "review_rate", "reject_rate"]]
        plot_frame.plot(
            kind="bar",
            stacked=True,
            color=["#4C78A8", "#F2CF5B", "#E45756"],
            ax=ax,
        )
        ax.set_xlabel("Policy Scenario")
        ax.set_ylabel("Population Share")
        ax.legend(loc="upper right")
        ax.tick_params(axis="x", rotation=0)
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def save_histogram(frame: pd.DataFrame, column: str, title: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(8, 4.8))
    values = pd.to_numeric(frame.get(column, pd.Series(dtype=float)), errors="coerce").dropna()
    if values.empty:
        ax.text(0.5, 0.5, "No data available", ha="center", va="center")
        ax.axis("off")
    else:
        sns.histplot(values, bins=35, kde=True, color="#4C78A8", ax=ax)
        ax.set_xlabel(column.replace("_", " ").title())
        ax.set_ylabel("Count")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def save_heatmap(
    matrix_frame: pd.DataFrame,
    row_column: str,
    column_column: str,
    value_column: str,
    title: str,
    path: Path,
    row_order: list[str] | None = None,
    column_order: list[str] | None = None,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(7.5, 6))
    if matrix_frame.empty:
        ax.text(0.5, 0.5, "No data available", ha="center", va="center")
        ax.axis("off")
    else:
        pivot = matrix_frame.pivot(index=row_column, columns=column_column, values=value_column).fillna(0.0)
        if row_order is not None:
            pivot = pivot.reindex(row_order)
        if column_order is not None:
            pivot = pivot.reindex(columns=column_order)
        sns.heatmap(pivot, annot=True, fmt=".0f", cmap="Blues", cbar=True, ax=ax)
        ax.set_xlabel(column_column.replace("_", " ").title())
        ax.set_ylabel(row_column.replace("_", " ").title())
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def save_group_policy_chart(
    summary_frame: pd.DataFrame,
    protected_attribute: str,
    title: str,
    path: Path,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    plot_frame = summary_frame.loc[
        summary_frame["protected_attribute"] == protected_attribute
    ].copy()
    plot_frame = _sort_group_metric_frame(plot_frame, protected_attribute)
    fig, ax = plt.subplots(figsize=(10, 4.8))
    if plot_frame.empty:
        ax.text(0.5, 0.5, "No data available", ha="center", va="center")
        ax.axis("off")
    else:
        x = np.arange(len(plot_frame))
        width = 0.38
        approved_bad = plot_frame["approved_bad_rate"].fillna(0.0)
        ax.bar(x - width / 2, plot_frame["approval_rate"], width=width, color="#4C78A8", label="Approval rate")
        ax.bar(x + width / 2, approved_bad, width=width, color="#E45756", label="Approved bad rate")
        ax.set_xticks(x)
        ax.set_xticklabels(plot_frame["group"].astype(str), rotation=35, ha="right")
        ax.set_ylabel("Rate")
        ax.legend(loc="best")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


print(f"Phase 5 output root: {PHASE_5_OUTPUT_ROOT}")
print(f"Phase 6 output root: {PHASE_6_OUTPUT_ROOT}")
print(f"Figure output dir: {FIGURE_OUTPUT_DIR}")
print(f"Default score settings: Base Score = {BASE_SCORE}, PDO = {POINTS_TO_DOUBLE_ODDS}, Base Odds = {BASE_ODDS}:1")
print(f"Policy scenarios: {POLICY_SCENARIOS}")


## 2. Readiness gate

Phase 6 默认依赖 `Phase 5` 的根产物：

- `candidate_model_selection.json`
- `validation_review_frame.csv`
- `summary.json`

如果这些产物不齐，先不要进入 score / cutoff 计算。


In [ ]:
required_root_files = {
    "candidate_model_selection": PHASE_5_OUTPUT_ROOT / "candidate_model_selection.json",
    "validation_review_frame": PHASE_5_OUTPUT_ROOT / "validation_review_frame.csv",
    "summary": PHASE_5_OUTPUT_ROOT / "summary.json",
}

readiness_frame = pd.DataFrame(
    [
        {
            "artifact": artifact_name,
            "path": str(path),
            "exists": path.exists(),
        }
        for artifact_name, path in required_root_files.items()
    ]
)
PHASE_6_READY = bool(readiness_frame["exists"].all())

display(readiness_frame)
display(pd.DataFrame([placeholder_metadata]))
print(f"Phase 6 ready: {PHASE_6_READY}")
print("PDO placeholder reminder: repository module remains scaffold-only; Phase 6 uses notebook-local score mapping.")

if not PHASE_6_READY:
    display(readiness_frame.loc[~readiness_frame["exists"]].reset_index(drop=True))
    print(
        "Phase 6 blocked. 请先完成 notebooks/05_xai_fairness.ipynb，并确认 candidate_model_selection / validation_review_frame / summary 已经生成。"
    )


## 3. 加载 candidate / comparator，并对齐 Phase 5 输出

这里把 Phase 5 的候选模型、matched comparator、proxy audit pair family 以及 grouped governance 列统一到一个 `score_input_frame`，为后续 score transform 和 cutoff 分析做准备。


In [ ]:
if not PHASE_6_READY:
    print("加载步骤已跳过，因为 Phase 5 根产物不完整。")
else:
    validation_errors = []
    phase5_selection_payload = json.loads(
        required_root_files["candidate_model_selection"].read_text(encoding="utf-8")
    )
    phase5_summary_payload = json.loads(
        required_root_files["summary"].read_text(encoding="utf-8")
    )
    validation_review_frame = pd.read_csv(required_root_files["validation_review_frame"])

    candidate_model = phase5_selection_payload.get("candidate_model") or {}
    matched_core_comparator = phase5_selection_payload.get("matched_core_comparator") or {}
    matched_proxy_comparator = phase5_selection_payload.get("matched_proxy_comparator") or {}

    candidate_feature_set = candidate_model.get("feature_set")
    candidate_model_family = candidate_model.get("model_family")
    proxy_audit_pair_family = phase5_selection_payload.get("proxy_audit_pair_family")

    if candidate_feature_set not in {"traditional_core", "traditional_plus_proxy"}:
        validation_errors.append("candidate_model.feature_set 不是受支持的 feature set。")
    if proxy_audit_pair_family not in {"logistic", "advanced"}:
        validation_errors.append("proxy_audit_pair_family 缺失或不受支持。")

    comparator_feature_set = (
        "traditional_core"
        if candidate_feature_set == "traditional_plus_proxy"
        else "traditional_plus_proxy"
    )
    comparator_model = (
        matched_core_comparator
        if comparator_feature_set == "traditional_core"
        else matched_proxy_comparator
    )

    core_score_column = f"{proxy_audit_pair_family}_traditional_core_predicted_pd"
    proxy_score_column = f"{proxy_audit_pair_family}_traditional_plus_proxy_predicted_pd"
    comparator_pd_column = (
        core_score_column if comparator_feature_set == "traditional_core" else proxy_score_column
    )

    required_columns = {
        settings.id_column,
        settings.target_column,
        "candidate_predicted_pd",
        core_score_column,
        proxy_score_column,
        *GROUP_REVIEW_COLUMNS,
    }
    missing_columns = sorted(required_columns - set(validation_review_frame.columns))
    if missing_columns:
        validation_errors.append(
            f"validation_review_frame 缺少列 {missing_columns}"
        )

    if validation_review_frame.empty:
        validation_errors.append("validation_review_frame 为空，无法开展 Phase 6 分析。")
    if settings.id_column in validation_review_frame.columns and validation_review_frame[settings.id_column].duplicated().any():
        validation_errors.append("validation_review_frame 主键不是唯一值。")

    if not validation_errors:
        selected_columns = [
            settings.id_column,
            settings.target_column,
            "candidate_predicted_pd",
            comparator_pd_column,
            *GROUP_REVIEW_COLUMNS,
        ]
        score_input_frame = validation_review_frame[selected_columns].copy()
        score_input_frame = score_input_frame.rename(
            columns={
                "candidate_predicted_pd": "candidate_pd",
                comparator_pd_column: "comparator_pd",
            }
        )
        score_input_frame[settings.target_column] = pd.to_numeric(
            score_input_frame[settings.target_column],
            errors="coerce",
        )
        score_input_frame["candidate_pd"] = pd.to_numeric(score_input_frame["candidate_pd"], errors="coerce")
        score_input_frame["comparator_pd"] = pd.to_numeric(score_input_frame["comparator_pd"], errors="coerce")
        score_input_frame["candidate_feature_set"] = candidate_feature_set
        score_input_frame["candidate_model_family"] = candidate_model_family
        score_input_frame["candidate_model_label"] = candidate_model.get("model_label")
        score_input_frame["comparator_feature_set"] = comparator_feature_set
        score_input_frame["comparator_model_family"] = comparator_model.get("model_family")
        score_input_frame["comparator_model_label"] = comparator_model.get("model_label")
        score_input_frame["proxy_audit_pair_family"] = proxy_audit_pair_family

        if score_input_frame[[settings.target_column, "candidate_pd", "comparator_pd"]].isna().any().any():
            validation_errors.append("score_input_frame 中存在无法解析的 target 或 predicted PD。")

    phase6_load_ok = len(validation_errors) == 0

    model_alignment_frame = pd.DataFrame(
        [
            {
                "role": "candidate_model",
                "model_label": candidate_model.get("model_label"),
                "model_family": candidate_model.get("model_family"),
                "feature_set": candidate_model.get("feature_set"),
            },
            {
                "role": "comparator_model",
                "model_label": comparator_model.get("model_label"),
                "model_family": comparator_model.get("model_family"),
                "feature_set": comparator_feature_set,
            },
        ]
    )
    display(model_alignment_frame)

    if candidate_model_family != proxy_audit_pair_family:
        print(
            "Warning: candidate_model.model_family 与 proxy_audit_pair_family 不一致。"
            "当前 Phase 6 将沿用 Phase 5 validation_review_frame 中的 candidate_predicted_pd。"
        )

    if not phase6_load_ok:
        display(pd.DataFrame({"validation_error": validation_errors}))
        print("Phase 6 停在输入对齐阶段，请先修复 Phase 5 artifact 或字段契约。")
    else:
        display(score_input_frame.head())
        print(f"Active feature set: {candidate_feature_set}")
        print(f"Candidate model label: {candidate_model.get('model_label')}")
        print(f"Comparator feature set: {comparator_feature_set}")


## 4. Score transform、风险分层与校准诊断

这一节把 candidate / comparator 的 predicted PD 映射到 score，并输出：

- A-E risk bands
- risk deciles
- calibration summary + Brier score
- Phase 6 的 score math / band integrity 基础校验


In [ ]:
if not PHASE_6_READY:
    print("评分映射步骤已跳过，因为 Phase 5 根产物不完整。")
elif not phase6_load_ok:
    print("评分映射步骤已跳过，因为输入对齐未通过。")
else:
    score_transform_meta = build_score_transform_meta(
        base_score=BASE_SCORE,
        points_to_double_odds=POINTS_TO_DOUBLE_ODDS,
        base_odds=BASE_ODDS,
    )

    score_frame = score_input_frame.copy()
    score_frame["candidate_score"] = pd_to_score(
        score_frame["candidate_pd"],
        score_transform_meta,
        name="candidate_score",
    )
    score_frame["comparator_score"] = pd_to_score(
        score_frame["comparator_pd"],
        score_transform_meta,
        name="comparator_score",
    )
    score_frame["candidate_score_minus_comparator"] = (
        score_frame["candidate_score"] - score_frame["comparator_score"]
    )
    score_frame["candidate_risk_band"] = assign_risk_band(score_frame["candidate_score"])
    score_frame["comparator_risk_band"] = assign_risk_band(score_frame["comparator_score"])
    score_frame["candidate_risk_decile"] = assign_score_decile(score_frame["candidate_score"])
    score_frame["comparator_risk_decile"] = assign_score_decile(score_frame["comparator_score"])

    score_decile_summary = summarize_score_deciles(
        score_frame,
        decile_column="candidate_risk_decile",
        score_column="candidate_score",
        pd_column="candidate_pd",
        target_column=settings.target_column,
    )
    risk_band_summary = summarize_risk_bands(
        score_frame,
        band_column="candidate_risk_band",
        score_column="candidate_score",
        pd_column="candidate_pd",
        target_column=settings.target_column,
    )
    calibration_summary = build_calibration_summary(
        score_frame,
        pd_column="candidate_pd",
        target_column=settings.target_column,
        score_column="candidate_score",
        bins=CALIBRATION_BINS,
    )

    score_check_rows = [
        {
            "check": "base_odds_maps_to_base_score",
            "passed": bool(np.isclose(odds_to_score(BASE_ODDS, score_transform_meta), BASE_SCORE, atol=1e-9)),
            "details": f"score({BASE_ODDS}:1) == {BASE_SCORE}",
        },
        {
            "check": "double_odds_adds_pdo_points",
            "passed": bool(
                np.isclose(
                    odds_to_score(BASE_ODDS * 2.0, score_transform_meta) - odds_to_score(BASE_ODDS, score_transform_meta),
                    POINTS_TO_DOUBLE_ODDS,
                    atol=1e-9,
                )
            ),
            "details": f"score({BASE_ODDS * 2:.1f}:1) - score({BASE_ODDS}:1) == {POINTS_TO_DOUBLE_ODDS}",
        },
        {
            "check": "higher_score_means_lower_pd",
            "passed": bool(pd_to_score(pd.Series([0.10]), score_transform_meta).iloc[0] > pd_to_score(pd.Series([0.20]), score_transform_meta).iloc[0]),
            "details": "score(10% PD) > score(20% PD)",
        },
        {
            "check": "risk_band_counts_match_score_frame",
            "passed": bool(int(risk_band_summary["count"].sum()) == len(score_frame)),
            "details": "sum(risk_band_summary.count) == len(score_frame)",
        },
        {
            "check": "risk_decile_counts_match_score_frame",
            "passed": bool(int(score_decile_summary["count"].sum()) == len(score_frame)),
            "details": "sum(score_decile_summary.count) == len(score_frame)",
        },
        {
            "check": "score_frame_has_unique_ids",
            "passed": bool(not score_frame[settings.id_column].duplicated().any()),
            "details": "SK_ID_CURR must remain unique after Phase 6 alignment.",
        },
        {
            "check": "brier_score_available",
            "passed": bool(not calibration_summary.empty and calibration_summary["brier_score"].notna().all()),
            "details": "Calibration summary must contain a finite Brier score.",
        },
    ]
    validation_checks = pd.DataFrame(score_check_rows)
    phase6_score_ok = bool(validation_checks["passed"].all())

    display(pd.DataFrame([score_transform_meta]))
    display(calibration_summary)
    display(score_decile_summary)
    display(risk_band_summary)
    display(validation_checks)
    print(f"Phase 6 score transform ready: {phase6_score_ok}")


## 5. Cutoff sweep、三段式策略与治理联动

这里同时做三类分析：

- 单阈值 `score_cutoff_grid` sweep
- `conservative / balanced / growth` 三段式策略
- candidate vs comparator 的 score / band / decision migration

同时把 `age_band`、`family_status_group`、`region_rating_group` 的 balanced policy grouped outcome 一起算出来。


In [ ]:
if not PHASE_6_READY:
    print("cutoff / 策略步骤已跳过，因为 Phase 5 根产物不完整。")
elif not phase6_score_ok:
    print("cutoff / 策略步骤已跳过，因为 score transform 或基础校验未通过。")
else:
    score_cutoff_grid = build_score_cutoff_grid(score_frame["candidate_score"], step=10)
    cutoff_sweep = build_cutoff_sweep(
        score_frame,
        score_column="candidate_score",
        pd_column="candidate_pd",
        target_column=settings.target_column,
        cutoff_grid=score_cutoff_grid,
    )

    policy_rows = []
    group_rows = []
    for scenario_name, thresholds in POLICY_SCENARIOS.items():
        candidate_decision_column = f"candidate_decision_{scenario_name}"
        comparator_decision_column = f"comparator_decision_{scenario_name}"
        score_frame[candidate_decision_column] = apply_policy(
            score_frame["candidate_score"],
            approve_cutoff=thresholds["approve_cutoff"],
            review_cutoff=thresholds["review_cutoff"],
        )
        score_frame[comparator_decision_column] = apply_policy(
            score_frame["comparator_score"],
            approve_cutoff=thresholds["approve_cutoff"],
            review_cutoff=thresholds["review_cutoff"],
        )
        policy_rows.append(
            build_policy_summary(
                score_frame,
                scenario_name=scenario_name,
                decision_column=candidate_decision_column,
                score_column="candidate_score",
                pd_column="candidate_pd",
                target_column=settings.target_column,
                approve_cutoff=thresholds["approve_cutoff"],
                review_cutoff=thresholds["review_cutoff"],
            )
        )
        group_rows.append(
            build_policy_group_summary(
                score_frame,
                group_columns=GROUP_REVIEW_COLUMNS,
                scenario_name=scenario_name,
                decision_column=candidate_decision_column,
                score_column="candidate_score",
                pd_column="candidate_pd",
                target_column=settings.target_column,
            )
        )

    policy_scenarios = pd.DataFrame(policy_rows)
    policy_group_summary = pd.concat(group_rows, ignore_index=True) if group_rows else pd.DataFrame()

    score_migration_matrix = build_migration_matrix(
        score_frame,
        from_column="comparator_risk_band",
        to_column="candidate_risk_band",
        count_column_name="count",
    )
    decision_migration_matrix = build_migration_matrix(
        score_frame,
        from_column="comparator_decision_balanced",
        to_column="candidate_decision_balanced",
        count_column_name="count",
    )

    policy_check_rows = []
    for scenario_row in policy_scenarios.to_dict(orient="records"):
        rate_total = (
            scenario_row["approval_rate"]
            + scenario_row["review_rate"]
            + scenario_row["reject_rate"]
        )
        count_total = (
            scenario_row["approved_count"]
            + scenario_row["review_count"]
            + scenario_row["rejected_count"]
        )
        policy_check_rows.extend(
            [
                {
                    "check": f"{scenario_row['scenario_name']}_policy_rates_sum_to_one",
                    "passed": bool(np.isclose(rate_total, 1.0, atol=1e-9)),
                    "details": "approval_rate + review_rate + reject_rate == 1",
                },
                {
                    "check": f"{scenario_row['scenario_name']}_policy_counts_cover_population",
                    "passed": bool(count_total == len(score_frame)),
                    "details": "approved_count + review_count + rejected_count == len(score_frame)",
                },
            ]
        )

    balanced_group_summary = policy_group_summary.loc[
        policy_group_summary["scenario_name"] == "balanced"
    ].copy()
    for protected_attribute in GROUP_REVIEW_COLUMNS:
        protected_count = int(
            balanced_group_summary.loc[
                balanced_group_summary["protected_attribute"] == protected_attribute,
                "count",
            ].sum()
        )
        policy_check_rows.append(
            {
                "check": f"balanced_group_summary_covers_population_{protected_attribute}",
                "passed": bool(protected_count == len(score_frame)),
                "details": "sum(group counts) for balanced scenario must equal len(score_frame)",
            }
        )

    policy_check_rows.extend(
        [
            {
                "check": "score_migration_matrix_covers_population",
                "passed": bool(int(score_migration_matrix["count"].sum()) == len(score_frame)),
                "details": "sum(score_migration_matrix.count) == len(score_frame)",
            },
            {
                "check": "decision_migration_matrix_covers_population",
                "passed": bool(int(decision_migration_matrix["count"].sum()) == len(score_frame)),
                "details": "sum(decision_migration_matrix.count) == len(score_frame)",
            },
            {
                "check": "score_cutoff_grid_not_empty",
                "passed": bool(len(score_cutoff_grid) > 0 and not cutoff_sweep.empty),
                "details": "score_cutoff_grid should contain at least one cutoff.",
            },
        ]
    )

    validation_checks = pd.concat(
        [validation_checks, pd.DataFrame(policy_check_rows)],
        ignore_index=True,
    )
    phase6_policy_ok = bool(validation_checks["passed"].all())

    display(cutoff_sweep.head(10))
    display(policy_scenarios)
    display(balanced_group_summary.head(30))
    display(score_migration_matrix)
    display(decision_migration_matrix)
    display(validation_checks)
    print(f"Phase 6 policy analysis ready: {phase6_policy_ok}")


## 6. 可视化、落盘与业务结论模板

按计划至少输出 14 张图，这里会固定生成 18 张图，以保证 score、band、cutoff、migration、grouped policy 的分析密度足够。


In [ ]:
if not PHASE_6_READY:
    print("可视化与落盘已跳过，因为 Phase 5 根产物不完整。")
elif not phase6_policy_ok:
    print("可视化与落盘已跳过，因为 Phase 6 计算或校验未通过。")
else:
    PHASE_6_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    candidate_model_label = str(candidate_model.get("model_label", "candidate_model")).replace("/", "_")
    output_dir = PHASE_6_OUTPUT_ROOT / candidate_model_label
    output_dir.mkdir(parents=True, exist_ok=True)

    balanced_group_summary = policy_group_summary.loc[
        policy_group_summary["scenario_name"] == "balanced"
    ].copy()

    figure_paths = {
        "score_histogram_kde": FIGURE_OUTPUT_DIR / f"phase6_{candidate_model_label}_score_histogram_kde.png",
        "score_ecdf": FIGURE_OUTPUT_DIR / f"phase6_{candidate_model_label}_score_ecdf.png",
        "pd_to_score_curve": FIGURE_OUTPUT_DIR / f"phase6_{candidate_model_label}_pd_to_score_curve.png",
        "calibration_curve": FIGURE_OUTPUT_DIR / f"phase6_{candidate_model_label}_calibration_curve.png",
        "decile_reliability": FIGURE_OUTPUT_DIR / f"phase6_{candidate_model_label}_decile_reliability.png",
        "risk_band_count": FIGURE_OUTPUT_DIR / f"phase6_{candidate_model_label}_risk_band_count.png",
        "risk_band_actual_default_rate": FIGURE_OUTPUT_DIR / f"phase6_{candidate_model_label}_risk_band_actual_default_rate.png",
        "risk_band_mean_predicted_pd": FIGURE_OUTPUT_DIR / f"phase6_{candidate_model_label}_risk_band_mean_predicted_pd.png",
        "cutoff_approval_rate": FIGURE_OUTPUT_DIR / f"phase6_{candidate_model_label}_cutoff_approval_rate.png",
        "cutoff_approved_bad_rate": FIGURE_OUTPUT_DIR / f"phase6_{candidate_model_label}_cutoff_approved_bad_rate.png",
        "cutoff_rejected_bad_capture_rate": FIGURE_OUTPUT_DIR / f"phase6_{candidate_model_label}_cutoff_rejected_bad_capture_rate.png",
        "policy_scenario_composition": FIGURE_OUTPUT_DIR / f"phase6_{candidate_model_label}_policy_scenario_composition.png",
        "score_delta_histogram": FIGURE_OUTPUT_DIR / f"phase6_{candidate_model_label}_score_delta_histogram.png",
        "band_migration_heatmap": FIGURE_OUTPUT_DIR / f"phase6_{candidate_model_label}_band_migration_heatmap.png",
        "decision_migration_heatmap": FIGURE_OUTPUT_DIR / f"phase6_{candidate_model_label}_decision_migration_heatmap.png",
        "age_band_balanced_policy": FIGURE_OUTPUT_DIR / f"phase6_{candidate_model_label}_age_band_balanced_policy.png",
        "family_status_balanced_policy": FIGURE_OUTPUT_DIR / f"phase6_{candidate_model_label}_family_status_balanced_policy.png",
        "region_rating_balanced_policy": FIGURE_OUTPUT_DIR / f"phase6_{candidate_model_label}_region_rating_balanced_policy.png",
    }

    save_score_histogram(
        score_frame,
        column="candidate_score",
        title=f"Phase 6 Score Distribution with KDE ({candidate_model_label})",
        path=figure_paths["score_histogram_kde"],
    )
    save_ecdf_curve(
        score_frame,
        column="candidate_score",
        title=f"Phase 6 Score ECDF ({candidate_model_label})",
        path=figure_paths["score_ecdf"],
    )
    save_pd_to_score_curve(
        score_transform_meta,
        title="Phase 6 PD-to-Score Transform Curve",
        path=figure_paths["pd_to_score_curve"],
    )
    save_calibration_curve(
        calibration_summary,
        title=f"Phase 6 Calibration Curve ({candidate_model_label})",
        path=figure_paths["calibration_curve"],
    )
    save_decile_reliability_chart(
        score_decile_summary,
        title=f"Phase 6 Decile Predicted vs Actual Default Rate ({candidate_model_label})",
        path=figure_paths["decile_reliability"],
    )
    save_risk_band_metric_chart(
        risk_band_summary,
        metric_column="count",
        title=f"Phase 6 Risk Band Distribution ({candidate_model_label})",
        path=figure_paths["risk_band_count"],
        color="#4C78A8",
    )
    save_risk_band_metric_chart(
        risk_band_summary,
        metric_column="actual_default_rate",
        title=f"Phase 6 Actual Default Rate by Risk Band ({candidate_model_label})",
        path=figure_paths["risk_band_actual_default_rate"],
        color="#E45756",
    )
    save_risk_band_metric_chart(
        risk_band_summary,
        metric_column="mean_predicted_pd",
        title=f"Phase 6 Mean Predicted PD by Risk Band ({candidate_model_label})",
        path=figure_paths["risk_band_mean_predicted_pd"],
        color="#72B7B2",
    )
    save_cutoff_metric_chart(
        cutoff_sweep,
        metric_column="approval_rate",
        title=f"Phase 6 Cutoff Sweep: Approval Rate ({candidate_model_label})",
        path=figure_paths["cutoff_approval_rate"],
    )
    save_cutoff_metric_chart(
        cutoff_sweep,
        metric_column="approved_bad_rate",
        title=f"Phase 6 Cutoff Sweep: Approved Bad Rate ({candidate_model_label})",
        path=figure_paths["cutoff_approved_bad_rate"],
    )
    save_cutoff_metric_chart(
        cutoff_sweep,
        metric_column="rejected_bad_capture_rate",
        title=f"Phase 6 Cutoff Sweep: Rejected Bad Capture Rate ({candidate_model_label})",
        path=figure_paths["cutoff_rejected_bad_capture_rate"],
    )
    save_policy_composition_chart(
        policy_scenarios,
        title=f"Phase 6 Policy Scenario Composition ({candidate_model_label})",
        path=figure_paths["policy_scenario_composition"],
    )
    save_histogram(
        score_frame,
        column="candidate_score_minus_comparator",
        title=f"Phase 6 Candidate Minus Comparator Score Delta ({candidate_model_label})",
        path=figure_paths["score_delta_histogram"],
    )
    save_heatmap(
        score_migration_matrix,
        row_column="comparator_risk_band",
        column_column="candidate_risk_band",
        value_column="count",
        title=f"Phase 6 Risk Band Migration ({candidate_model_label})",
        path=figure_paths["band_migration_heatmap"],
        row_order=RISK_BAND_ORDER,
        column_order=RISK_BAND_ORDER,
    )
    save_heatmap(
        decision_migration_matrix,
        row_column="comparator_decision_balanced",
        column_column="candidate_decision_balanced",
        value_column="count",
        title=f"Phase 6 Balanced Decision Migration ({candidate_model_label})",
        path=figure_paths["decision_migration_heatmap"],
        row_order=DECISION_ORDER,
        column_order=DECISION_ORDER,
    )
    save_group_policy_chart(
        balanced_group_summary,
        protected_attribute="age_band",
        title=f"Phase 6 Balanced Policy by Age Band ({candidate_model_label})",
        path=figure_paths["age_band_balanced_policy"],
    )
    save_group_policy_chart(
        balanced_group_summary,
        protected_attribute="family_status_group",
        title=f"Phase 6 Balanced Policy by Family Status ({candidate_model_label})",
        path=figure_paths["family_status_balanced_policy"],
    )
    save_group_policy_chart(
        balanced_group_summary,
        protected_attribute="region_rating_group",
        title=f"Phase 6 Balanced Policy by Region Rating ({candidate_model_label})",
        path=figure_paths["region_rating_balanced_policy"],
    )

    score_frame.to_csv(output_dir / "score_frame.csv", index=False)
    score_decile_summary.to_csv(output_dir / "score_decile_summary.csv", index=False)
    risk_band_summary.to_csv(output_dir / "risk_band_summary.csv", index=False)
    calibration_summary.to_csv(output_dir / "calibration_summary.csv", index=False)
    cutoff_sweep.to_csv(output_dir / "cutoff_sweep.csv", index=False)
    policy_scenarios.to_csv(output_dir / "policy_scenarios.csv", index=False)
    policy_group_summary.to_csv(output_dir / "policy_group_summary.csv", index=False)
    score_migration_matrix.to_csv(output_dir / "score_migration_matrix.csv", index=False)
    decision_migration_matrix.to_csv(output_dir / "decision_migration_matrix.csv", index=False)
    (output_dir / "score_transform_meta.json").write_text(
        json.dumps(_to_builtin(score_transform_meta), indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    summary_payload = {
        "candidate_model": candidate_model,
        "comparator_model": comparator_model,
        "proxy_audit_pair_family": phase5_selection_payload.get("proxy_audit_pair_family"),
        "score_settings": score_transform_meta,
        "placeholder_pdo_metadata": placeholder_metadata,
        "calibration_fitted": False,
        "score_cutoff_grid": score_cutoff_grid,
        "brier_score": float(calibration_summary["brier_score"].iloc[0]) if not calibration_summary.empty else None,
        "risk_band_summary": risk_band_summary.to_dict(orient="records"),
        "policy_scenarios": policy_scenarios.to_dict(orient="records"),
        "validation_checks": validation_checks.to_dict(orient="records"),
        "figure_paths": {key: str(path) for key, path in figure_paths.items()},
        "phase5_business_conclusion_template": phase5_summary_payload.get("business_conclusion_template"),
        "top_phase5_proxy_uplift_rows": phase5_summary_payload.get("top_proxy_uplift_rows", [])[:5],
        "limitations": [
            "Repository PDO module is still placeholder-only; score transform is notebook-local analysis logic.",
            "Phase 6 runs calibration diagnostics only and does not fit Platt or isotonic calibration.",
            "Grouped policy outputs remain governance diagnostics only, not a production-ready fairness audit.",
        ],
    }
    (output_dir / "summary.json").write_text(
        json.dumps(_to_builtin(summary_payload), indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    saved_outputs = pd.DataFrame(
        [
            {"artifact": "score_frame", "path": str(output_dir / "score_frame.csv")},
            {"artifact": "score_transform_meta", "path": str(output_dir / "score_transform_meta.json")},
            {"artifact": "score_decile_summary", "path": str(output_dir / "score_decile_summary.csv")},
            {"artifact": "risk_band_summary", "path": str(output_dir / "risk_band_summary.csv")},
            {"artifact": "calibration_summary", "path": str(output_dir / "calibration_summary.csv")},
            {"artifact": "cutoff_sweep", "path": str(output_dir / "cutoff_sweep.csv")},
            {"artifact": "policy_scenarios", "path": str(output_dir / "policy_scenarios.csv")},
            {"artifact": "policy_group_summary", "path": str(output_dir / "policy_group_summary.csv")},
            {"artifact": "score_migration_matrix", "path": str(output_dir / "score_migration_matrix.csv")},
            {"artifact": "decision_migration_matrix", "path": str(output_dir / "decision_migration_matrix.csv")},
            {"artifact": "summary", "path": str(output_dir / "summary.json")},
        ]
    )
    display(saved_outputs)
    print(f"Saved figure count: {len(figure_paths)}")


## 7. Checkpoint

结束前集中检查：

- Phase 6 是否通过 readiness / score / policy 三层校验
- 评分映射是否明确标注为 notebook-local
- calibration / governance 限制是否写入 summary


In [ ]:
if not PHASE_6_READY:
    print("Checkpoint: Phase 6 仍被阻断，请先补齐 Phase 5 根产物。")
elif not phase6_load_ok:
    print("Checkpoint: Phase 6 仍停在输入对齐，请先修复 candidate / comparator 或 validation_review_frame 字段。")
elif not phase6_score_ok:
    print("Checkpoint: Phase 6 仍停在 score transform / calibration，请先修复基础校验。")
elif not phase6_policy_ok:
    print("Checkpoint: Phase 6 仍停在 cutoff / policy / migration，请先修复策略与迁移校验。")
else:
    print("Phase 6 scorecard + cutoff analysis complete.")
    print(f"Candidate model: {candidate_model.get('model_label')}")
    print(f"Active feature set: {candidate_model.get('feature_set')}")
    print(f"Proxy audit pair family: {phase5_selection_payload.get('proxy_audit_pair_family')}")
    print(f"Generated figures: {len(figure_paths)}")
    print("重要限制：repository PDO 模块仍是 placeholder；当前分数映射仅用于分析，不代表 production-ready scorecard。")
    print("校准限制：Phase 6 只做 calibration diagnostics，没有拟合新的校准器。")
    print("治理限制：grouped policy 结果只代表 governance diagnostic，不代表完整 fairness audit。")
    display(policy_scenarios)
    display(validation_checks)


In [ ]:
# Phase 6 report upgrade add-ons
from credit_visable.governance import build_group_fairness_metric_summary
from credit_visable.scoring import build_profit_assumption_config, select_optimal_profit_threshold
from credit_visable.utils import (
    REPORT_COLOR_PALETTE,
    add_conclusion_annotation,
    build_report_summary_fields,
    format_percent_axis,
    to_builtin,
)

profit_curve = pd.DataFrame()
optimal_profit_cutoff = {}
optimal_profit_fairness_summary = pd.DataFrame()

if not PHASE_6_READY:
    print("Phase 6 report upgrade skipped because Phase 5 root artifacts are incomplete.")
elif not phase6_policy_ok:
    print("Phase 6 report upgrade skipped because score, cutoff, or policy checks are incomplete.")
else:
    candidate_model_label = str(candidate_model.get("model_label", "candidate_model")).replace("/", "_")
    output_dir = PHASE_6_OUTPUT_ROOT / candidate_model_label
    output_dir.mkdir(parents=True, exist_ok=True)
    FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    profit_assumptions = build_profit_assumption_config()
    profit_rows = []
    total_bad = float(score_frame[settings.target_column].sum())
    for cutoff in score_cutoff_grid:
        approved_mask = score_frame["candidate_score"] >= cutoff
        rejected_mask = ~approved_mask
        approved_good_count = int((approved_mask & (score_frame[settings.target_column] == 0)).sum())
        approved_bad_count = int((approved_mask & (score_frame[settings.target_column] == 1)).sum())
        rejected_good_count = int((rejected_mask & (score_frame[settings.target_column] == 0)).sum())
        rejected_bad_count = int((rejected_mask & (score_frame[settings.target_column] == 1)).sum())
        total_profit = (
            approved_good_count * profit_assumptions["approve_good"]
            + approved_bad_count * profit_assumptions["approve_bad"]
            + rejected_good_count * profit_assumptions["reject_good"]
            + rejected_bad_count * profit_assumptions["reject_bad"]
        )
        profit_rows.append(
            {
                "score_cutoff": int(cutoff),
                "approval_rate": float(approved_mask.mean()),
                "reject_rate": float(rejected_mask.mean()),
                "approved_bad_rate": float(score_frame.loc[approved_mask, settings.target_column].mean()) if approved_mask.any() else np.nan,
                "rejected_bad_capture_rate": float(score_frame.loc[rejected_mask, settings.target_column].sum() / total_bad) if total_bad > 0 else 0.0,
                "total_profit": float(total_profit),
                "profit_per_applicant": float(total_profit / len(score_frame)) if len(score_frame) > 0 else 0.0,
            }
        )
    profit_curve = pd.DataFrame(profit_rows)
    optimal_profit_cutoff = select_optimal_profit_threshold(
        profit_curve.rename(columns={"score_cutoff": "threshold"})
    )
    optimal_profit_cutoff["score_cutoff"] = int(optimal_profit_cutoff.pop("threshold"))
    balanced_profit_row = profit_curve.loc[
        profit_curve["score_cutoff"] == POLICY_SCENARIOS["balanced"]["approve_cutoff"]
    ]
    if balanced_profit_row.empty:
        balanced_profit_row = profit_curve.iloc[[int(np.argmin(np.abs(profit_curve["score_cutoff"] - POLICY_SCENARIOS["balanced"]["approve_cutoff"])))]]
    balanced_profit_row = balanced_profit_row.iloc[0]

    cutoff_sweep = cutoff_sweep.drop(columns=["total_profit", "profit_per_applicant"], errors="ignore")
    cutoff_sweep = cutoff_sweep.merge(
        profit_curve[["score_cutoff", "total_profit", "profit_per_applicant"]],
        on="score_cutoff",
        how="left",
    )
    cutoff_sweep.to_csv(output_dir / "cutoff_sweep.csv", index=False)
    profit_curve.to_csv(output_dir / "profit_curve.csv", index=False)
    (output_dir / "profit_assumptions.json").write_text(
        json.dumps(to_builtin(profit_assumptions), indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    (output_dir / "optimal_profit_cutoff.json").write_text(
        json.dumps(to_builtin(optimal_profit_cutoff), indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    def score_to_pd(score_value, meta):
        odds = meta["base_odds"] * np.exp((score_value - meta["base_score"]) / meta["factor"])
        return 1.0 / (1.0 + odds)

    optimal_pd_threshold = score_to_pd(optimal_profit_cutoff["score_cutoff"], score_transform_meta)
    optimal_profit_fairness_summary = build_group_fairness_metric_summary(
        frame=score_frame,
        target_column=settings.target_column,
        score_column="candidate_pd",
        group_specs=[
            {
                "protected_attribute": column,
                "source_column": column,
                "group_column": column,
                "kind": "identity",
            }
            for column in GROUP_REVIEW_COLUMNS
        ],
        threshold=float(optimal_pd_threshold),
    )
    optimal_profit_fairness_summary.to_csv(output_dir / "optimal_profit_fairness_summary.csv", index=False)

    phase6_extra_figure_paths = {
        "profit_curve": FIGURE_OUTPUT_DIR / f"phase6_{candidate_model_label}_cutoff_profit_curve.png",
    }
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(profit_curve["score_cutoff"], profit_curve["total_profit"], color=REPORT_COLOR_PALETTE["good"], linewidth=2.2)
    ax.axvline(POLICY_SCENARIOS["balanced"]["approve_cutoff"], linestyle="--", color=REPORT_COLOR_PALETTE["neutral"], label="Balanced cutoff")
    ax.axvline(optimal_profit_cutoff["score_cutoff"], linestyle="--", color=REPORT_COLOR_PALETTE["highlight"], label="Optimal profit cutoff")
    ax.set_title(f"Phase 6 Profit Curve ({candidate_model_label})")
    ax.set_xlabel("Score cutoff")
    ax.set_ylabel("Total profit")
    profit_delta = float(optimal_profit_cutoff["total_profit"] - balanced_profit_row["total_profit"])
    add_conclusion_annotation(ax, f"Profit delta vs balanced: {profit_delta:.2f}")
    ax.legend(loc="best")
    fig.tight_layout()
    fig.savefig(phase6_extra_figure_paths["profit_curve"], dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    figure_paths.update(phase6_extra_figure_paths)
    summary_path = output_dir / "summary.json"
    existing_summary = json.loads(summary_path.read_text(encoding="utf-8")) if summary_path.exists() else {}
    top_fairness_row = optimal_profit_fairness_summary.sort_values(
        "equalized_odds_gap", ascending=False
    ).iloc[0] if not optimal_profit_fairness_summary.empty else None
    existing_summary.update(
        build_report_summary_fields(
            headline="Phase 6 now adds profit-based cutoff optimization alongside score, band, migration, and calibration diagnostics.",
            key_findings=[
                f"Optimal profit cutoff: {optimal_profit_cutoff['score_cutoff']}.",
                f"Balanced cutoff profit: {balanced_profit_row['total_profit']:.2f}.",
                f"Largest fairness sensitivity at optimal cutoff: {top_fairness_row['protected_attribute']}" if top_fairness_row is not None else "Largest fairness sensitivity at optimal cutoff: n/a",
            ],
            business_implications=[
                "Phase 6 can now discuss profitability directly instead of relying only on approval and bad-rate trade-offs.",
                "The balanced scenario remains unchanged, but it is now benchmarked against an explicit profit-optimal cutoff.",
                "Fairness sensitivity at the optimal cutoff is visible, which helps prevent purely economic threshold decisions.",
            ],
            figure_paths={path.stem: path for path in sorted(FIGURE_OUTPUT_DIR.glob(f"phase6_{candidate_model_label}_*.png"))},
        )
    )
    existing_summary["profit_assumptions"] = profit_assumptions
    existing_summary["optimal_profit_cutoff"] = optimal_profit_cutoff
    existing_summary["balanced_cutoff_profit"] = to_builtin(balanced_profit_row.to_dict())
    existing_summary["optimal_profit_fairness_summary"] = optimal_profit_fairness_summary.to_dict(orient="records")
    summary_path.write_text(json.dumps(to_builtin(existing_summary), indent=2, ensure_ascii=False), encoding="utf-8")

    display(profit_curve.head(10))
    display(optimal_profit_fairness_summary)
